# Shikibu DL の学習（Colab 用、M11）

NNUE の教師データ（PSV）で、方策と価値の 2 つの頭を持つ DL を学習します（`docs/GUIDE_COLAB.md` の DL の節）。
- 学習は裏のプロセス（`shikibu.train_dl`）で走るので、表示のセルを止めても、ブラウザを再読み込みしても続きます。
- 途中で切れても、同じ手順で続きから再開できます。1 回の実行は `max_minutes`（既定 8 時間）で正常に終わります。
- 学習した重みの書き出し（Dainagon のエンジンへ載せる形式）は、ONNX の書き出し（M11.1）が決まってから行います。
  それまでは `checkpoint.pt` が学習の成果物です。
- 同じ出力フォルダで 2 つの学習が同時に動くことはありません（ロック `train.lock`）。

## 1. Google Drive をマウントする

Drive にあるリポジトリ・wheel・データ・出力を使うため、最初にマウントします。

In [ ]:
# Google Drive を /content/drive にマウントする（Colab でだけ使える）
from google.colab import drive
drive.mount('/content/drive')


## 2. 設定（ここだけ書き換える）

`<…>` の目印を、自分の環境の値に書き換えてください。目印が残っていると、次のセルでエラーになります。

- `REPO_URL`・`BRANCH`：リポジトリの取得元。Drive に `REPO` が無いときだけ使う。
- `REPO`：Drive 上のリポジトリの場所。
- `WHEEL_DIR`：`shk_feat` の wheel の保存先（無ければ初回にビルドして保存する）。
- `DATA_GLOB`：学習用の PSV のパス（`*.psv` のように glob で書く）。
- `OUT_ROOT`・`RUN_NAME`：出力は `OUT_ROOT/RUN_NAME` に作られる。
- `NETWORK`：プリセット `A`（幅重視）・`B`（深さ重視）・`X2`（約 2 倍）、または `dl_…` の名前を直接書く。
- `CACHE_DIR`：Colab のローカルの一時コピー先（Drive には置かない）。
- `SESSION_MAX_STEPS`：このセッションで区切る通算の更新回数。`None` なら設定の `max_steps`。

In [ ]:
# ===== 設定：<…> の目印を、自分の環境の値に書き換える =====
REPO_URL = "https://github.com/<GitHub のユーザー名>/Shikibu.git"   # リポジトリの取得元（Drive に無いときだけ使う）
BRANCH = "main"                                                    # 取得するブランチ
REPO = "/content/drive/MyDrive/<リポジトリを置くフォルダ>/Shikibu"     # Drive 上のリポジトリの場所
WHEEL_DIR = "/content/drive/MyDrive/<wheel を置くフォルダ>"             # shk_feat の wheel の保存先
DATA_GLOB = "/content/drive/MyDrive/<学習データのフォルダ>/*.psv"        # 学習用の PSV（組を 1 つだけ使うときの場所）
OUT_ROOT = "/content/drive/MyDrive/<出力のフォルダ>"                     # 学習の出力の親フォルダ
RUN_NAME = "<学習の名前>"                                               # 出力フォルダ名（OUT_ROOT/RUN_NAME）
NETWORK = "A"                                                        # プリセット A・B・X2、または dl_… の名前
TEMPLATE = "configs/colab_dl.toml"                                   # ひな形の設定
CACHE_DIR = ""                                                       # 空なら Drive から直接読む（既定。大きいデータで cache_dir を使うと待つ）
SESSION_MAX_STEPS = None                                             # None なら設定の max_steps。数値なら通算の更新回数の上限

# ----- 組（データの混ぜ方）。指し手のあるデータと、評価値・勝敗だけのデータを混ぜるときに使う（M11.1）-----
# 組を 1 つだけ使うなら、SOURCES = None にして DATA_GLOB を使う。
# 例：指し手のある hcpe（方策・勝敗を学ぶ）と、指し手の無い大量の PSV（価値・勝敗を学ぶ）を半々に混ぜる。
SOURCES = None
# SOURCES = [
#     dict(name="moves_hcpe", glob="/content/drive/MyDrive/<指し手のあるデータ>/*.hcpe", format="hcpe",
#          weight=0.5, policy=True, value=False, result=True, eval_scale="auto"),
#     dict(name="big_psv", glob="/content/drive/MyDrive/<評価値と勝敗だけのデータ>/*.psv", format="psv",
#          weight=0.5, policy=False, value=True, result=True, eval_scale="auto"),
# ]
VAL_PATH = ""            # 検証用の PSV（学習に使わないファイル）。空なら psv の検証はしない
VAL_HCPE_GLOB = ""       # 検証用の hcpe（Dainagon のテストセット）。空なら hcpe の検証はしない
BATCH_SIZE = 4096        # 1 回の更新の局面数
AMP = "auto"             # "auto"：T4 なら "fp16"、それ以外の CUDA は "bf16"。CPU は "off"
EVAL_SCALE = "auto"      # 組ごとに推定するなら "auto"。数値なら全組その値（組の設定があれば、そちらが優先）
TOTAL_STEPS = None       # None なら設定のまま（ひな形は 35 万）。学習率の減衰の長さ
MAX_MINUTES = 480        # 1 回の実行の上限（分）。0 なら無制限
VAL_EVERY_STEPS = 0      # 検証の間隔（更新回数）。0 なら終わりに 1 回
SHUFFLE_BUFFER = None    # None なら設定のまま（ひな形は 2,000 万）

# ----- テスト用の既定（True なら、30 分・5000 ステップで回す。動くことを確かめるための設定）-----
QUICK_TEST = True
if QUICK_TEST:
    MAX_MINUTES = 30
    TOTAL_STEPS = 5000
    VAL_EVERY_STEPS = 500
    VAL_POSITIONS = 20000
    SHUFFLE_BUFFER = 1000000
    SESSION_MAX_STEPS = None
else:
    VAL_POSITIONS = 100000

# ===== 目印の確認（ここで止まったら、上の <…> を書き換える） =====
_settings = {"REPO_URL": REPO_URL, "REPO": REPO, "WHEEL_DIR": WHEEL_DIR, "DATA_GLOB": DATA_GLOB,
             "OUT_ROOT": OUT_ROOT, "RUN_NAME": RUN_NAME}
_left = [k for k, v in _settings.items() if "<" in v and ">" in v]
if SOURCES is not None:
    _left += [f"SOURCES[{i}].glob" for i, s in enumerate(SOURCES) if "<" in s["glob"] and ">" in s["glob"]]
if _left:
    raise ValueError("設定セルの <…> を書き換えてください: " + ", ".join(_left))
print("設定を確認しました。", "（テスト用の既定：30 分・5000 ステップ）" if QUICK_TEST else "")

## 3. リポジトリを取得（または更新）する

`REPO` があれば `git pull` で更新し、無ければ `REPO_URL` から `git clone` します。

In [ ]:
import os
import subprocess
import glob

os.makedirs(WHEEL_DIR, exist_ok=True)
if os.path.isdir(os.path.join(REPO, ".git")):
    subprocess.run(["git", "-C", REPO, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO, "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO, "pull", "origin", BRANCH], check=True)
else:
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, REPO], check=True)
print("リポジトリ:", REPO, "/ ブランチ:", BRANCH)

## 4. wheel（`shikibu_feat`）をインストールする

- `WHEEL_DIR` に `shk_feat-*.whl` があれば、それを入れます（数秒）。
- 無ければ Rust を入れて `maturin build --release` で wheel を作り、`WHEEL_DIR` に保存してから入れます（初回だけ）。
- Rust のソースを変えたときは、`WHEEL_DIR` の wheel を消してから実行し直してください。
- DL の学習は `dl_record`・`BatchLoader(output="dl")` を使うので、M11 以降の wheel が要ります。

In [ ]:
# 既存の wheel を探す。複数あれば一番新しいものを使う
wheels = sorted(glob.glob(os.path.join(WHEEL_DIR, "shk_feat-*.whl")), key=os.path.getmtime)
# M11.1 より前の wheel（dl_record_hcpe も、指し手のない局面を残す機能も無い）は使わない：消して作り直す
if wheels and subprocess.run(["python", "-c", "import shikibu_feat as f; assert hasattr(f, 'dl_record_hcpe')"],
                             capture_output=True).returncode != 0:
    for w in wheels:
        os.remove(w)
    print("古い wheel（M11.1 より前）を消しました。作り直します。")
    wheels = []

if wheels:
    subprocess.run(["pip", "install", "-q", "--force-reinstall", wheels[-1]], check=True)
    print("保存済みの wheel を入れました:", wheels[-1])
else:
    if subprocess.run(["bash", "-lc", "command -v cargo"], capture_output=True).returncode != 0:
        subprocess.run("curl -sSf https://sh.rustup.rs | sh -s -- -y", shell=True, check=True)
    cargo_env = os.path.expanduser("~/.cargo/bin")
    env = dict(os.environ, PATH=cargo_env + os.pathsep + os.environ["PATH"])
    subprocess.run(["pip", "install", "-q", "maturin"], check=True)
    subprocess.run(["maturin", "build", "--release", "-m", "crates/shk_feat/Cargo.toml",
                    "-o", WHEEL_DIR], cwd=REPO, env=env, check=True)
    built = sorted(glob.glob(os.path.join(WHEEL_DIR, "shk_feat-*.whl")), key=os.path.getmtime)
    subprocess.run(["pip", "install", "-q", "--force-reinstall", built[-1]], check=True)
    print("wheel をビルドして保存しました:", built[-1])

import torch
import shikibu_feat
print("torch", torch.__version__, "/ CUDA:", torch.cuda.is_available(), "/ GPU:",
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "なし")
print("論理コア数:", os.cpu_count(), "/ shikibu_feat OK, dl_record:", hasattr(shikibu_feat, "dl_record"))

## 5. 設定ファイルを出力フォルダに書き出す

ひな形（`TEMPLATE`）の `<…>` の値を、上の設定セルの値で置き換えて、`OUT_ROOT/RUN_NAME/config.toml` に書き出します。
学習はこの書き出したファイルを使います。Drive に残るので、再開のときもそのまま使えます。

In [ ]:
import json
import re
import tomllib

def set_key(text, key, value):
    """ひな形の `key = …` の行を、value の値に置き換える（行はちょうど1つでなければならない）。"""
    if isinstance(value, bool):
        lit = "true" if value else "false"
    elif isinstance(value, str):
        lit = '"%s"' % value
    else:
        lit = repr(value)
    new, n = re.subn(rf"^{key} = .*$", f"{key} = {lit}", text, flags=re.M)
    if n != 1:
        raise ValueError(f"ひな形に {key} の行が1つだけ無い（{n} 個）")
    return new

PRESETS = {"A": "dl_192_SHHHHT_H_HH", "B": "dl_128_SSHHHHHHHHTT_H_HH", "X2": "dl_192_SSHHHHHHTT_H_HH"}
NETWORK_NAME = PRESETS.get(NETWORK, NETWORK)
OUT_DIR = os.path.join(OUT_ROOT, RUN_NAME)
text = open(os.path.join(REPO, TEMPLATE), encoding="utf-8").read()
text = set_key(text, "train_glob", DATA_GLOB)
text = set_key(text, "cache_dir", CACHE_DIR)
text = set_key(text, "out_dir", OUT_DIR)
text = set_key(text, "network", NETWORK_NAME)
if SESSION_MAX_STEPS is not None:
    text = set_key(text, "max_steps", SESSION_MAX_STEPS)
text = set_key(text, "cache_dir", CACHE_DIR)
text = set_key(text, "batch_size", BATCH_SIZE)
text = set_key(text, "val_path", VAL_PATH)
text = set_key(text, "val_hcpe_glob", VAL_HCPE_GLOB)
text = set_key(text, "max_minutes", MAX_MINUTES)
text = set_key(text, "val_every_steps", VAL_EVERY_STEPS)
text = set_key(text, "val_positions", VAL_POSITIONS)
text = set_key(text, "eval_scale", EVAL_SCALE if EVAL_SCALE == "auto" else float(EVAL_SCALE))
text = set_key(text, "amp", AMP if AMP != "auto" else ("bf16" if __import__("torch").cuda.is_available() and "T4" not in __import__("torch").cuda.get_device_name(0) else ("fp16" if __import__("torch").cuda.is_available() else "off")))
if TOTAL_STEPS is not None:
    text = set_key(text, "total_steps", TOTAL_STEPS)
if SHUFFLE_BUFFER is not None:
    text = set_key(text, "shuffle_buffer", SHUFFLE_BUFFER)
if SOURCES is not None:
    # 組は [[data.sources]] の表で書く（ひな形の末尾に足す。ひな形の train_glob は、組があるときは使われない）
    lines = ["", "# ----- 組（ノートブックの SOURCES から書き出した）-----"]
    for s in SOURCES:
        lines.append("[[data.sources]]")
        for k, v in s.items():
            lines.append(f"{k} = {json.dumps(v, ensure_ascii=False)}")
        lines.append("")
    text = text.rstrip("\n") + "\n" + "\n".join(lines)

os.makedirs(OUT_DIR, exist_ok=True)
CONFIG = os.path.join(OUT_DIR, "config.toml")
with open(CONFIG, "w", encoding="utf-8") as f:
    f.write(text)
cfg = tomllib.loads(text)

shards = sorted(glob.glob(DATA_GLOB))
print("学習用の PSV:", len(shards), "個")
for p in shards:
    print("  ", os.path.basename(p), "%.2f GB" % (os.path.getsize(p) / 1e9))
print("設定:", CONFIG)
print("構造:", NETWORK_NAME)
print("出力先:", OUT_DIR, "/ checkpoint:", "あり（--resume で続きから）" if os.path.exists(os.path.join(OUT_DIR, "checkpoint.pt")) else "なし（最初から）")
print("バッチ:", cfg["train"]["batch_size"], "/ 総更新回数:", cfg["train"]["max_steps"], "/ シャッフルバッファ:", cfg["data"]["shuffle_buffer"])

## 6. 学習を裏で開始する（`mon.start`）

- `checkpoint.pt` があれば `--resume` を付けて続きから学習します。無ければ最初から学習します。
- 同じ出力フォルダで学習が実行中なら、学習器はロックで断り、次の節の表示に理由が出ます。
- 切断の直後で、ロックが残ったまま学習が動いていないとき（`docs/OPERATIONS.md` の 3）だけ、`force=True` にします。
- 止めるときは `mon.stop()` を実行します（最後の保存より後の分は失われます）。

In [ ]:
import sys

sys.path.insert(0, os.path.join(REPO, "tools"))
import shk_monitor as mon

FORCE_UNLOCK = False   # 切断の直後で、学習が動いていないのにロックが残っているときだけ True にする
mon.start(CONFIG, repo=REPO, force=FORCE_UNLOCK)
if "CONFIG" not in globals():
    print("CONFIG が無い：節 5（設定ファイルを書き出す）を実行するか、CONFIG に設定ファイルのパスを書いてください。")
else:
    mon.start(CONFIG, repo=REPO, force=FORCE_UNLOCK)


## 7. 進み具合を表示する（`mon.watch`）

15秒ごとに、段階・進捗・残り時間・速さ・学習曲線を表示します。学習の間は、このセルを動かし続けてください。

In [ ]:
if "CONFIG" not in globals():
    print("CONFIG が無い：節 5 を実行するか、CONFIG に設定ファイルのパスを書いてください。")
else:
    mon.watch(CONFIG)

## 8. 速さを測る（任意）

学習の前に、プリセットごとの速さを測れます（読み込み・展開・1 ステップ）。学習中に動かすと、学習の速さが落ちるので、学習の前に行ってください。

In [ ]:
import subprocess
SPEED = subprocess.run(["python", os.path.join(REPO, "tools", "shk_speedcheck_dl.py"), "--repo", REPO,
                        "--glob", DATA_GLOB, "--presets", "A,B,X2", "--batch", str(BATCH_SIZE),
                        "--threads", str(cfg["data"]["num_threads"]), "--steps", "20", "--shift", "--blocks"],
                       capture_output=True, text=True)
print(SPEED.stdout)
print(SPEED.stderr[-2000:] if SPEED.returncode else "")